# 04. SIR Evaluation: Semantic Irrelevance Ratio

**Purpose:** Compute the Semantic Irrelevance Ratio (SIR) for GCR, DCA-Trie v1, and DCA-Trie v2 from saved predictions.

**What SIR measures:** At each decoding step t, what fraction of paths in the valid token set are semantically irrelevant to the question? SIR is independent of answer accuracy — it measures the constraint oracle's quality, not the LLM's.

**Reference:** Thesis §3.4. Equations (3.5)-(3.7).

**Formula:**
```
SIR(q, t) = (number of paths in trie at step t with cos_sim < τ_ref) / (total paths in trie at step t)
```
where τ_ref = 0.3 is the fixed reference threshold.

Corpus SIR = average over all questions × decoding steps.

In [ ]:
# @title 1. Setup
import sys, os, json, torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

!pip install -q sentence-transformers datasets marisa-trie networkx

if not os.path.exists("graph-constrained-reasoning"):
    !git clone https://github.com/rmanluo/graph-constrained-reasoning.git
%cd graph-constrained-reasoning
sys.path.insert(0, '.')

from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import src.utils as utils

In [ ]:
# @title 2. Configuration
# Reference threshold for SIR
TAU_REF = 0.3   # τ_ref — fixed, NOT the admission threshold

# Sentence encoder (must match what was used in DCA-Trie)
ENCODER_NAME = "all-MiniLM-L6-v2"
ENCODER_DEVICE = "cpu"

# Path to raw dataset (to reconstruct graph + paths)
DATA_PATH = "rmanluo"
DATASET = "RoG-webqsp"
SPLIT = "test"

# Point to each system's predictions
MODEL_SHORT = "GCR-Meta-Llama-3.1-8B-Instruct"

EXPERIMENTS = {
    "GCR": {
        "path": f"results/GenPaths/{DATASET}/{MODEL_SHORT}/{SPLIT}/zero-shot-group-beam-k5-index_len2/predictions.jsonl",
    },
    "DCA-Trie v1 (τ=0.25)": {
        "path": f"results/GenPaths/{DATASET}/{MODEL_SHORT}/{SPLIT}/zero-shot-group-beam-k5-index_len2-DCAv1-tau0.25-all-MiniLM-L6-v2/predictions.jsonl",
    },
    "DCA-Trie v2 (τ=0.25)": {
        "path": f"results/GenPaths/{DATASET}/{MODEL_SHORT}/{SPLIT}/zero-shot-group-beam-k5-index_len2-DCAv2-tau0.25-all-MiniLM-L6-v2/predictions.jsonl",
    },
}

print(f"Reference threshold τ_ref: {TAU_REF}")
print(f"Encoder: {ENCODER_NAME}")
print(f"Dataset: {DATASET}")
for name, cfg in EXPERIMENTS.items():
    exists = os.path.exists(cfg["path"])
    print(f"  {name}: {'✅ found' if exists else '❌ not found'} at {cfg['path']}")

In [ ]:
# @title 3. Load Encoder and Dataset
print("Loading encoder...")
encoder = SentenceTransformer(ENCODER_NAME, device=ENCODER_DEVICE)

print("Loading dataset...")
dataset = load_dataset(f"{DATA_PATH}/{DATASET}", split=SPLIT)
data_by_id = {d["id"]: d for d in dataset}
print(f"Loaded {len(data_by_id)} questions")

In [ ]:
# @title 4. SIR Computation
def compute_sir_for_experiment(predictions_path, encoder, data_by_id, tau_ref, index_len=2):
    """
    Compute SIR for a single experiment.

    For each question:
      1. Reconstruct the KG-Trie (same paths the model saw during decoding)
      2. For each step t (1 to max_hops):
         a. Get all paths in the trie at that step
         b. Score each against (question + generated_prefix)
         c. SIR(q,t) = count(score < τ_ref) / total_paths
      3. Corpus SIR = mean over all q × t
    """
    if not os.path.exists(predictions_path):
        print(f"  File not found: {predictions_path}")
        return None

    with open(predictions_path) as f:
        predictions = [json.loads(l) for l in f]

    all_sirs = []          # per-question × per-step SIR values
    step_sirs = defaultdict(list)  # SIR grouped by hop depth
    total_trie_sizes = []
    irrelevant_counts = []

    for pred in tqdm(predictions, desc="Computing SIR"):
        qid = pred["id"]
        if qid not in data_by_id:
            continue
        data = data_by_id[qid]
        question = pred["question"]

        # Reconstruct graph
        g = utils.build_graph(data["graph"])

        # Reconstruct the trie paths (same as what the system used)
        if "paths" in data:
            paths_list = data["paths"]
        else:
            paths_list = utils.dfs(g, data["q_entity"], index_len)

        if len(paths_list) == 0:
            continue

        path_strs = [utils.path_to_string(p) for p in paths_list]
        if len(path_strs) == 0:
            continue

        # Encode all paths once (caching)
        path_embeddings = encoder.encode(path_strs, convert_to_numpy=True)

        # Encode the question (used as context at every step for GCR)
        q_emb = encoder.encode(question, convert_to_numpy=True)

        # For each hop depth, compute SIR
        # (In GCR, all paths are valid at every step, so we compute SIR per hop depth)
        for step in range(1, index_len + 1):
            # Build the generated prefix (we may not have it, so use question alone)
            # For baseline SIR, context = question (no generated prefix available)
            # For DCA-Trie v2, context = question + partial generation
            context = question

            # Score all paths
            sims = cosine_similarity(q_emb.reshape(1, -1), path_embeddings)[0]

            total = len(sims)
            irrelevant = int(np.sum(sims < tau_ref))
            sir_step = irrelevant / total if total > 0 else 0

            all_sirs.append(sir_step)
            step_sirs[step].append(sir_step)
            total_trie_sizes.append(total)
            irrelevant_counts.append(irrelevant)

    results = {
        "corpus_sir": float(np.mean(all_sirs)),
        "corpus_sir_std": float(np.std(all_sirs)),
        "avg_trie_size": float(np.mean(total_trie_sizes)),
        "avg_irrelevant": float(np.mean(irrelevant_counts)),
        "num_questions": len(predictions),
        "num_observations": len(all_sirs),
    }

    # Per-hop SIR
    for step in sorted(step_sirs.keys()):
        results[f"sir_hop_{step}"] = float(np.mean(step_sirs[step]))

    return results

In [ ]:
# @title 5. Compute SIR for All Experiments
all_results = {}
for name, cfg in EXPERIMENTS.items():
    print(f"\n{'='*60}")
    print(f"Computing SIR for: {name}")
    print(f"  {cfg['path']}")
    results = compute_sir_for_experiment(
        cfg["path"], encoder, data_by_id, TAU_REF, INDEX_LEN
    )
    all_results[name] = results

In [ ]:
# @title 6. Results Table
print("\n" + "="*80)
print("SEMANTIC IRRELEVANCE RATIO (SIR) — Comparison")
print("="*80)
print(f"Reference threshold τ_ref = {TAU_REF}")
print(f"Dataset: {DATASET}")
print()

header = f"{'System':<25} {'Corpus SIR':<12} {'Avg Trie Size':<15} {'SIR hop-1':<10} {'SIR hop-2':<10} {'Questions':<10}"
print(header)
print("-" * len(header))

for name, results in all_results.items():
    if results is None:
        print(f"{name:<25} {'N/A':<12}")
        continue
    sir = results["corpus_sir"]
    size = results["avg_trie_size"]
    sir1 = results.get("sir_hop_1", 0)
    sir2 = results.get("sir_hop_2", 0)
    n = results["num_questions"]
    print(f"{name:<25} {sir:<12.4f} {size:<15.1f} {sir1:<10.4f} {sir2:<10.4f} {n:<10}")

In [ ]:
# @title 7. Analysis
print("""
=== INTERPRETATION ===

SIR measures what fraction of the valid token set is semantically irrelevant
at each step. Lower SIR = tighter, more focused constraints.

Expected pattern:
  GCR:           highest SIR (all structurally valid paths allowed, many irrelevant)
  DCA-Trie v1:   lower SIR (pre-filters against question semantics)
  DCA-Trie v2:   lowest SIR (conditions on question + generation state)

Key insight: SIR is independent of answer accuracy. A system can have high Hits@1
but also high SIR (the LLM finds the right answer despite a noisy constraint set).
DCA-Trie aims to reduce SIR without sacrificing Hits@1.
""")

# Also compute the reduction
if "GCR" in all_results and all_results["GCR"]:
    gcr_sir = all_results["GCR"]["corpus_sir"]
    for name, results in all_results.items():
        if name == "GCR" or results is None:
            continue
        reduction = (gcr_sir - results["corpus_sir"]) / gcr_sir * 100
        print(f"{name}: SIR reduction vs GCR = {reduction:.1f}%")